In [11]:
from sklearn.decomposition import TruncatedSVD
import nltk
import pandas as pd
import numpy as np
import random
import re
import wordcloud
from nltk.corpus import stopwords
from nltk.corpus import movie_reviews
import sklearn.feature_extraction
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import string
from wordcloud import WordCloud
from collections import Counter

In [13]:
def create_corpus(filename):
    df = pd.read_csv(filename)
    return df["text"].dropna().tolist()

corpus = create_corpus("../data/train.csv")
corpus

["Licitació de la concessió administrativa de l'ús privatiu del domini públic municipal d'una peça de terreny.",
 "Notificació de l'acord de deixar sense efecte l'aprovació definitiva d'un projecte de reparcel·lació.",
 "Aprovació de l'expedient de dissolució i liquidació de la Junta de Compensació de Can Poi del Bosc de la Garriga.",
 "Aprovació inicial de la modificació puntual dels usos admesos de la clau 7 del Pla d'ordenació urbanística municipal de Cardedeu i del PMU-24 carrer Mogent.",
 "Aprovació definitiva de l'avanç del Pla de millora de la Franja Nord.",
 'Aprovació definitiva del Pla de millora del subsector 1 de la Franja Nord.',
 "Convocatòria per a la cessió gratuïta d'ordinadors PC declarats bé no utilitzable de l'Ajuntament.",
 "Aprovació definitiva de la modificació de les bases generals que regulen la concessió d'ajuts a les famílies.",
 "Subvencions atorgades l'any 2012 d'import igual o superior a 3.000 euros.",
 "Aprovació definitiva de les bases per sol·licitar i 

## Cleanup and preprocessing

Now that we have our corpus created, we may clean it to make it easy to analyze and obtain information. To do that, we have followed these steps:

* First, turn everything into lowercase
* Then, remove numbers, as they will not give rellevant info on topics

In [18]:
def text_cleanup(corpus):
    stop_words = set(stopwords.words('catalan'))
    cleaned_corpus = []
    
    for doc in corpus:
        doc = doc.lower()
        doc = re.sub(r'\b(?!19\d{2}|20\d{2})\d+\b', '', doc)
        doc = re.sub(r'[^\w\s]', '', doc)
        doc = re.sub(r'\b_+\b', '', doc)
        doc = re.sub(r'\s+', ' ', doc).strip()
        doc = ' '.join([word for word in doc.split() if word not in stop_words])
        cleaned_corpus.append(doc)
    
    return cleaned_corpus

cleaned_corpus = text_cleanup(corpus)
cleaned_corpus

['licitació concessió administrativa lús privatiu domini públic municipal duna peça terreny',
 'notificació lacord deixar efecte laprovació definitiva dun projecte reparcellació',
 'aprovació lexpedient dissolució liquidació junta compensació can poi bosc garriga',
 'aprovació inicial modificació puntual usos admesos clau pla dordenació urbanística municipal cardedeu pmu carrer mogent',
 'aprovació definitiva lavanç pla millora franja nord',
 'aprovació definitiva pla millora subsector franja nord',
 'convocatòria cessió gratuïta dordinadors pc declarats utilitzable lajuntament',
 'aprovació definitiva modificació bases generals regulen concessió dajuts famílies',
 'subvencions atorgades lany 2012 dimport superior euros',
 'aprovació definitiva bases sollicitar atorgar subvencions rehabilitació equipaments elements collectius edificis lexercici 2012',
 'subvencions superiors tres mil euros atorgades diferents serveis làrea presidència',
 'notificació dunes resolucions sancionadores',
 